# 4.3 The Base Classification Model

Every classifier we build from here on (softmax regression, MLPs, CNNs, ...)
shares the same plumbing: run a validation step and — unlike regression —
report **accuracy**, not just loss. Rather than repeat that in every model, we
factor it into a base `Classifier` class, subclassed from `d2l.Module`, and
reuse it throughout the book.

This section defines no new model. It is pure infrastructure: a `validation_step`,
a default optimizer, and an `accuracy` metric.

In [1]:
import torch
import torch.nn.functional as F
from torch import nn
from d2l import torch as d2l

## 4.3.1 The `Classifier` Class

The base class only needs to override `validation_step`: run the model on a
batch, then log both the **loss** and the **accuracy** on the validation curve.

A batch is a tuple, so `batch[:-1]` are the inputs (usually just `X`) and
`batch[-1]` is the label vector `y`. Unpacking with `*` means the same code
works for models that take more than one input tensor. `train=False` tells the
`ProgressBoard` to draw these on the validation curve rather than the training
one.

In [2]:
class Classifier(d2l.Module):  #@save
    """The base class of classification models."""
    def validation_step(self, batch):
        Y_hat = self(*batch[:-1])                                        # forward pass -> logits
        self.plot('loss', self.loss(Y_hat, batch[-1]), train=False)      # log val loss
        self.plot('acc',  self.accuracy(Y_hat, batch[-1]), train=False)  # log val accuracy

Note what `Classifier` does *not* define: `forward`, `loss`, or any parameters.
Those belong to the concrete model — that is the subclass's job in §4.4 and
§4.5. `validation_step` simply assumes they exist.

We also give every `d2l.Module` a default optimizer — plain minibatch **SGD**
over all parameters — so subclasses don't have to repeat it. `self.parameters()`
comes from `nn.Module` and collects every learnable tensor; `self.lr` was stored
by `save_hyperparameters()` in the subclass constructor.

In [3]:
@d2l.add_to_class(d2l.Module)  #@save
def configure_optimizers(self):
    return torch.optim.SGD(self.parameters(), lr=self.lr)   # vanilla SGD, learning rate self.lr

## 4.3.2 Accuracy

Accuracy is the fraction of examples predicted correctly. Given the logits
$\hat{\mathbf{y}}$ for one example, the hard prediction is the class with the
largest score:

$$\hat{y} = \operatorname*{argmax}_j \hat{y}_j$$

Three details are worth pausing on:

- **Reshape first.** If `Y_hat` arrives with extra leading dimensions, flattening
  to `(num_examples, num_classes)` makes the rest of the code dimension-agnostic.
- **`==` is dtype-sensitive.** `argmax` returns `int64`, but `Y` may be stored
  differently, so we cast `preds` to `Y.dtype` before comparing.
- **Cast the booleans to float** before averaging — the mean of a `bool` tensor
  is not defined in PyTorch.

Accuracy is *not differentiable* (argmax has zero gradient almost everywhere,
and is undefined at ties), so we cannot optimize it directly. We minimize the
smooth cross-entropy loss instead and *report* accuracy as the human-meaningful
number.

In [4]:
@d2l.add_to_class(Classifier)  #@save
def accuracy(self, Y_hat, Y, averaged=True):
    """Compute the fraction of correct predictions."""
    Y_hat = Y_hat.reshape((-1, Y_hat.shape[-1]))            # flatten to (num_examples, num_classes)
    preds = Y_hat.argmax(axis=1).type(Y.dtype)              # predicted class = index of largest logit
    compare = (preds == Y.reshape(-1)).type(torch.float32)  # 1.0 where correct, else 0.0
    return compare.mean() if averaged else compare          # mean -> accuracy in [0, 1]

A worked example makes the shapes concrete. Four examples, three classes, so
`Y_hat` is $4 \times 3$ and `Y` has length 4. Passing `averaged=False` returns
the per-example 0/1 vector, which is useful when you want to accumulate counts
across batches yourself.

In [5]:
Y_hat = torch.tensor([[0.1, 0.9, 0.0],    # -> class 1
                      [2.0, 0.5, 0.3],    # -> class 0
                      [0.2, 0.1, 0.7],    # -> class 2
                      [1.0, 1.5, 0.4]])   # -> class 1
Y = torch.tensor([1, 0, 1, 1])            # ground truth: the third one is wrong

clf = Classifier()
print('predictions:', Y_hat.argmax(axis=1))
print('per-example :', clf.accuracy(Y_hat, Y, averaged=False))   # 1 = correct, 0 = wrong
print('accuracy    :', clf.accuracy(Y_hat, Y))                   # 3 of 4 -> 0.75

predictions: tensor([1, 0, 2, 1])
per-example : tensor([1., 1., 0., 1.])
accuracy    : tensor(0.7500)


Notice that we never called softmax. Softmax is **monotonically increasing**, so
it cannot change which entry is largest — normalizing just to pick the winner
would be wasted work. The probabilities differ from the logits, but the `argmax`
is identical.

In [6]:
probs = F.softmax(Y_hat, dim=1)
print('probabilities:\n', probs)
print('rows sum to  :', probs.sum(dim=1))
print('argmax logits:', Y_hat.argmax(axis=1))
print('argmax probs :', probs.argmax(axis=1))
print('same ranking :', torch.equal(Y_hat.argmax(axis=1), probs.argmax(axis=1)))

probabilities:
 tensor([[0.2421, 0.5388, 0.2191],
        [0.7113, 0.1587, 0.1299],
        [0.2814, 0.2546, 0.4640],
        [0.3127, 0.5156, 0.1716]])
rows sum to  : tensor([1., 1., 1., 1.])
argmax logits: tensor([1, 0, 2, 1])
argmax probs : tensor([1, 0, 2, 1])
same ranking : True


## 4.3.3 A Sanity Check on Real Data

Before training anything, it is worth confirming the plumbing works end to end.
We build a minimal `Classifier` subclass — a flatten plus one linear layer, the
shape of the softmax regression model we implement in §4.4 — and run a single
Fashion-MNIST batch through it **without any training**.

`nn.LazyLinear` infers its input size on the first forward pass, so we only have
to state the number of outputs (10 classes).

In [7]:
class UntrainedClassifier(Classifier):
    """Softmax-regression-shaped model with random, untrained weights."""
    def __init__(self, num_outputs=10, lr=0.1):
        super().__init__()
        self.save_hyperparameters()                    # stores num_outputs and lr
        self.net = nn.Sequential(nn.Flatten(),         # (batch, 1, 28, 28) -> (batch, 784)
                                 nn.LazyLinear(num_outputs))   # -> (batch, 10) logits

    def forward(self, X):
        return self.net(X)

    def loss(self, Y_hat, Y):
        return F.cross_entropy(Y_hat, Y)

In [8]:
torch.manual_seed(42)
data = d2l.FashionMNIST(batch_size=256)
X, y = next(iter(data.val_dataloader()))     # one validation minibatch

model = UntrainedClassifier()
with torch.no_grad():                        # inference only: no gradients needed
    Y_hat = model(X)

print('logits shape :', Y_hat.shape)                       # (256, 10)
print('loss         : %.4f' % model.loss(Y_hat, y))        # ~ln(10) = 2.3026
print('ln(10)       : %.4f' % torch.tensor(10.0).log())
print('accuracy     : %.4f' % model.accuracy(Y_hat, y))    # ~0.1 = random guessing

logits shape : torch.Size([256, 10])
loss         : 2.3381
ln(10)       : 2.3026
accuracy     : 0.0703


Both numbers are exactly what an untrained model should produce. With random
weights the predicted distribution is near uniform, so the cross-entropy sits
close to $\ln 10 \approx 2.303$ and accuracy sits near $1/10$ — this run's
256-example batch lands at $7.0\%$, comfortably inside the sampling noise a
batch that small produces around a true rate of $10\%$. These are the
baselines to beat — if a *trained* classifier ever reports numbers like
these, it has learned nothing.

We call `loss` and `accuracy` directly here rather than `validation_step`,
because `validation_step` calls `self.plot`, which needs a `Trainer` attached to
the model. The `Trainer` supplies that in §4.4.

### The fine print on averaging

The `Trainer` reports validation loss by averaging the *per-batch* means —
call this quick estimate $L_\textrm{v}^\textrm{q}$. It differs from the *true*
validation loss $L_\textrm{v}$, the mean over all individual examples, whenever
the last minibatch has a different size than the rest. This is exactly the
setup behind the book's first two exercises for this section: with the loss
on that last minibatch denoted $l_\textrm{v}^\textrm{b}$, the cell below
reconstructs $L_\textrm{v}$ from $L_\textrm{v}^\textrm{q}$, $l_\textrm{v}^\textrm{b}$,
and the batch sizes alone, and checks the reconstruction against the ground
truth.

In [9]:
torch.manual_seed(0)
n, b = 1000, 256
per_example = torch.rand(n)                     # pretend per-example validation losses
batches = per_example.split(b)                  # sizes: 256, 256, 256, 232
sizes = [len(batch) for batch in batches]
m, r = len(batches), sizes[-1]                   # m batches total, r = size of the last one
print('batch sizes            :', sizes)

batch_means = torch.stack([batch.mean() for batch in batches])
true_mean  = per_example.mean()           # L_v:   true average, over all n examples
quick_mean = batch_means.mean()           # L_v^q: Trainer's average of the batch means
last_batch = batch_means[-1]              # l_v^b: loss on the last (short) minibatch

# Exercise 1: reconstruct L_v from L_v^q, l_v^b, and the sizes alone.
reconstructed = (b * m / n) * quick_mean - (b - r) / n * last_batch

print('true  average  L_v     : %.6f' % true_mean)
print('quick average  L_v^q   : %.6f' % quick_mean)
print('reconstructed  L_v     : %.6f' % reconstructed)
print('|L_v - L_v^q|          : %.6f' % (true_mean - quick_mean).abs())

batch sizes            : [256, 256, 256, 232]
true  average  L_v     : 0.500925
quick average  L_v^q   : 0.501093
reconstructed  L_v     : 0.500925
|L_v - L_v^q|          : 0.000168


**Exercise 1.** With $n$ validation examples split into $m$ batches of size
$b$ — all but the last, which has size $r = n - (m-1)b$ — the definitions
above rearrange into

$$L_\textrm{v} = \frac{bm}{n}\,L_\textrm{v}^\textrm{q} \;-\; \frac{b-r}{n}\,l_\textrm{v}^\textrm{b}.$$

`reconstructed` verifies this numerically: it matches `true_mean` to
floating-point precision using only the two aggregate numbers
$L_\textrm{v}^\textrm{q}$ and $l_\textrm{v}^\textrm{b}$ plus the batch sizes —
never the individual per-example losses in between.

**Exercise 2.** The `DataLoader` shuffles examples before splitting them into
batches, so every batch is a uniformly random subset of the validation set.
For simple random sampling, the mean of *any* fixed-size random subset is
itself an unbiased estimator of the population mean, so
$E[\bar\ell_j] = L_\textrm{v}$ for every batch $j$ — and an average of unbiased
estimators is unbiased, so $E[L_\textrm{v}^\textrm{q}] = E[L_\textrm{v}]$ too
($L_\textrm{v}$ is fixed given the validation set, so its expectation is just
itself).

Unbiasedness is not the whole story, though, which is why the book still
prefers $L_\textrm{v}$: it weights every example by exactly $1/n$, so it is
*invariant* to how the shuffle happens to split the data into batches — group
it however you like and you get the same number back. $L_\textrm{v}^\textrm{q}$
instead lets whichever examples land in the short final batch outweigh their
peers, so its value wobbles from shuffle to shuffle — the `0.000168` gap above
comes purely from *how* the same 1000 losses were grouped. That gap is tiny
here because the batches are large and few; it vanishes entirely once the
batch size is chosen to divide the validation set evenly.

### Loss and accuracy are different objectives

The book's third exercise for this section asks for the optimal prediction
rule under a general multiclass loss $l(y, y')$: given the true class
posterior $p(y \mid x)$, choose the $y'$ that minimizes the *expected* loss,

$$y' = \operatorname*{argmin}_{y'} \sum_{y} p(y \mid x)\, l(y, y').$$

Plug in the 0-1 loss $l(y, y') = \mathbb{1}(y \neq y')$ and this reduces to
$y' = \operatorname*{argmax}_y p(y \mid x)$ — exactly the rule that maximizes
accuracy. Cross-entropy is a *different* loss, one that keeps caring about the
whole probability vector, not just which entry is largest. So although both
objectives share the same optimum once $p(y \mid x)$ is known exactly,
minimizing cross-entropy during training and maximizing accuracy on a finite
validation set need not move together: a gradient step can lower the loss by
making an already-correct prediction more confident, while accuracy — which
only sees the argmax — stays completely unmoved. Two logit vectors for the
same example make it concrete: both predict the correct class, so both score
identically on accuracy, yet their cross-entropy losses differ by nearly
$28\times$.

In [10]:
y = torch.tensor([0])                                # true label: class 0
logits_confident = torch.tensor([[4.0, 0.0, 0.0]])   # very sure, and right
logits_timid     = torch.tensor([[0.6, 0.5, 0.4]])   # barely sure, and right

print('argmax        :', logits_confident.argmax(1).item(), logits_timid.argmax(1).item())
print('accuracy      :', clf.accuracy(logits_confident, y).item(), clf.accuracy(logits_timid, y).item())
print('cross-entropy : %.4f  vs  %.4f' % (F.cross_entropy(logits_confident, y), F.cross_entropy(logits_timid, y)))

argmax        : 0 0
accuracy      : 1.0 1.0
cross-entropy : 0.0360  vs  1.0019


That is the sense in which loss minimization and accuracy maximization need
not coincide: cross-entropy keeps rewarding confidence long after argmax has
already found the right answer, and keeps punishing a confident mistake far
more than a timid one, while accuracy is blind to all of it. Minimizing the
loss is a good proxy for maximizing accuracy — it is not the same target.

## 4.3.4 Summary

- **`Classifier`** is a thin `d2l.Module` subclass that logs both loss and
  accuracy at validation time — reused by every classification model to come.
  It defines no parameters, forward pass, or loss; subclasses supply those.
- **Prediction** = `argmax` over the logits. Softmax is monotonic, so there is
  no need to normalize just to pick the winner.
- **Accuracy** = mean of `(prediction == label)`. It is the metric we *report*,
  while training optimizes the differentiable cross-entropy loss instead —
  argmax has no useful gradient.
- Watch the **dtypes**: cast predictions to the label dtype before `==`, and
  cast the resulting booleans to float before averaging.
- A default **SGD** optimizer is attached to `d2l.Module`, so subclasses only
  need to define their architecture and loss.
- An untrained 10-class model should score $\approx \ln 10$ loss and $\approx 10\%$
  accuracy — the baseline that §4.4 sets out to beat.
- The `Trainer`'s validation loss $L_\textrm{v}^\textrm{q}$ (an average of
  batch means) is an unbiased but higher-variance stand-in for the true
  per-example average $L_\textrm{v}$; the two coincide exactly once the batch
  size divides the validation set.
- **Loss and accuracy are related but distinct objectives**: minimizing
  cross-entropy is a good proxy for maximizing accuracy, not an identical
  target — argmax, and therefore accuracy, is blind to confidence changes
  that the loss keeps reacting to.